xác định path Bronze Weather Forecast

In [0]:
from fastorder.storage.adls_client import get_adls_service_client
service_client = get_adls_service_client()

In [0]:
bronze_client = service_client.get_file_system_client(file_system="bronze")
for path in bronze_client.get_paths(path = "weather/open_meteo", recursive = True):
    print(path.name)

Khám phá Bronze Weather Forecast

In [0]:
import os

tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")
account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")

endpoint = f"{account_name}.dfs.core.windows.net"

spark.conf.set(
    f"fs.azure.account.auth.type.{endpoint}",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{endpoint}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{endpoint}",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{endpoint}",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{endpoint}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
relative_metadata_path = "weather/open_meteo/forecast/ingestion_date=2026-08-16/warehouse_id=WH_CT/ingestion_id=1573a177-27db-5ba6-8fb7-ac91844cc521/metadata.json"
# spark_path = abfss://<filesystem>@<storage-account>.dfs.core.windows.net/<path-bên-trong-filesystem>
spark_metadata_path = f"abfss://bronze@fastorderdatalake.dfs.core.windows.net/{relative_path}"

In [0]:
df = spark.read.option("multiline", True).json(spark_metadata_path)
display(df)

In [0]:
relative_response_path = "weather/open_meteo/forecast/ingestion_date=2026-08-16/warehouse_id=WH_CT/ingestion_id=1573a177-27db-5ba6-8fb7-ac91844cc521/response.json"

spark_response_path = f"abfss://bronze@fastorderdatalake.dfs.core.windows.net/{relative_response_path}"
df = spark.read.option("multiline", True).json(spark_response_path)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql import functions as F

weather_df = df.select(
    "latitude",
    "longitude",
    "timezone",
    F.explode(
        F.arrays_zip(
            "hourly.time",
            "hourly.temperature_2m",
            "hourly.relative_humidity_2m",
            "hourly.precipitation",
            "hourly.wind_speed_10m",
            "hourly.weather_code",
        )
    ).alias("weather")
)

In [0]:
weather_df = weather_df.select(
    "latitude",
    "longitude",
    "timezone",
    F.col("weather.time").alias("time"),
    F.col("weather.temperature_2m").alias("temperature_2m"),
    F.col("weather.relative_humidity_2m").alias("relative_humidity_2m"),
    F.col("weather.precipitation").alias("precipitation"),
    F.col("weather.wind_speed_10m").alias("wind_speed_10m"),
    F.col("weather.weather_code").alias("weather_code"),
)

In [0]:
weather_df.display()

In [0]:
from pyspark.sql.functions import count
cnt_temperature_2m = weather_df.groupBy("latitude").agg(count("temperature_2m"))
cnt_temperature_2m.display()